In [ ]:
!pip install pythainlp gensim matplotlib --quiet

In [ ]:
RANDOM_STATE = 42
BATCH_SIZE = 128
LR = 2e-4
EPOCHS = 100

HIDDEN_DIM = 300
MAX_LEN = 512
MAX_VOCAB = 20000

PAD_ID = 0
UNK_ID = 1

In [ ]:
labels = [
    "politics", "human_rights", "quality_of_life", "international",
    "social", "environment", "economics", "culture", "labor",
    "national_security", "ict", "education"
]

NUM_CLASSES = len(labels)

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from collections import Counter

from pythainlp.tokenize import word_tokenize
from pythainlp.util import normalize
from pythainlp.word_vector import WordVector

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
path = "/kaggle/input/pythainlp-prachatai-67k"

In [ ]:
train_df = pd.read_csv(f"{path}/train.csv").dropna()
val_df   = pd.read_csv(f"{path}/validation.csv").dropna()
test_df  = pd.read_csv(f"{path}/test.csv").dropna()

train_df = train_df[['body_text'] + labels]
val_df   = val_df[['body_text'] + labels]
test_df  = test_df[['body_text'] + labels]

In [ ]:
def process_th(text):
    text = normalize(str(text))
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def tokenize_th(text):
    tokens = word_tokenize(text, engine="newmm", keep_whitespace=False)
    return [t for t in tokens if len(t.strip()) > 1]

train_df['text'] = train_df['body_text'].apply(process_th).apply(tokenize_th)
val_df['text']   = val_df['body_text'].apply(process_th).apply(tokenize_th)
test_df['text']  = test_df['body_text'].apply(process_th).apply(tokenize_th)

In [ ]:
counter = Counter()
for t in train_df['text']:
    counter.update(t)

vocab = {"<pad>": PAD_ID, "<unk>": UNK_ID}
for i, (w, _) in enumerate(counter.most_common(MAX_VOCAB - 2), start=2):
    vocab[w] = i

def encode(tokens):
    if len(tokens) > MAX_LEN:
        half = MAX_LEN // 2
        tokens = tokens[:half] + tokens[-half:]
    ids = [vocab.get(t, UNK_ID) for t in tokens]
    return ids + [PAD_ID] * (MAX_LEN - len(ids))

vocab_size = len(vocab)

In [ ]:
print("Loading Thai2Vec...")
thai2vec = WordVector()

embedding_weight = np.zeros((vocab_size, HIDDEN_DIM), dtype=np.float32)
rng = np.random.default_rng(RANDOM_STATE)

for word, idx in vocab.items():
    if word in ("<pad>", "<unk>"):
        continue
    try:
        embedding_weight[idx] = thai2vec.get_vector(word)
    except:
        embedding_weight[idx] = rng.normal(0, 0.01, size=(HIDDEN_DIM,))

In [ ]:
class TextDataset(Dataset):
    def __init__(self, df):
        self.texts = df['text'].tolist()
        self.labels = df[labels].values.astype(np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return (
            torch.tensor(encode(self.texts[idx]), dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )

train_loader = DataLoader(TextDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TextDataset(val_df), batch_size=BATCH_SIZE)
test_loader  = DataLoader(TextDataset(test_df), batch_size=BATCH_SIZE)

In [ ]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, output_dim, embedding_matrix=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)

        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(torch.tensor(embedding_matrix, dtype=torch.float32))

        self.rnn = nn.LSTM(embed_dim, embed_dim // 2, batch_first=True)

        self.fc1 = nn.Linear(embed_dim // 2, embed_dim // 4)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(embed_dim // 4, output_dim)

    def forward(self, text):
        lengths = (text != PAD_ID).sum(dim=1)

        embedded = self.embedding(text)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        _, (hidden, _) = self.rnn(packed)

        out = hidden[-1]

        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)

        return out

model = RNN(vocab_size, HIDDEN_DIM, NUM_CLASSES, embedding_weight).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.BCEWithLogitsLoss()

In [ ]:
@torch.no_grad()
def evaluate(loader, threshold=0.5):
    model.eval()
    preds, labels_ = [], []

    for x, y in loader:
        x = x.to(DEVICE)
        logits = model(x)

        probs = torch.sigmoid(logits)
        pred = (probs > threshold).int().cpu().numpy()

        preds.append(pred)
        labels_.append(y.numpy())

    y_pred = np.vstack(preds)
    y_true = np.vstack(labels_)

    # ✅ Hamming Accuracy
    acc = (y_pred == y_true).mean()

    exact_acc = (y_pred == y_true).all(axis=1).mean()

    f1_micro = f1_score(y_true, y_pred, average="micro")
    f1_macro = f1_score(y_true, y_pred, average="macro")

    return acc, exact_acc, f1_micro, f1_macro

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    acc, exact_acc, f1_micro, f1_macro = evaluate(val_loader)

    print(
        f"Epoch {epoch+1:03d} | "
        f"loss={total_loss:.6f} | "
        f"hamming_acc={acc:.6f} | "
        f"exact_acc={exact_acc:.6f} | "
        f"f1_micro={f1_micro:.6f} | "
        f"f1_macro={f1_macro:.6f}"
    )

In [ ]:
acc, exact_acc, f1_micro, f1_macro = evaluate(test_loader)

print(
    f"\nFINAL TEST:\n"
    f"Hamming Accuracy = {acc:.6f}\n"
    f"Exact Match Acc = {exact_acc:.6f}\n"
    f"F1-micro        = {f1_micro:.6f}\n"
    f"F1-macro        = {f1_macro:.6f}"
)

# =========================
# PREDICT
# =========================
@torch.no_grad()
def predict(text, threshold=0.5):
    tokens = tokenize_th(process_th(text))
    x = torch.tensor([encode(tokens)], dtype=torch.long).to(DEVICE)

    probs = torch.sigmoid(model(x)).cpu().numpy()[0]
    pred = (probs > threshold)

    return {
        "labels": [labels[i] for i in range(NUM_CLASSES) if pred[i]],
        "scores": {labels[i]: float(probs[i]) for i in range(NUM_CLASSES)}
    }

print(predict("รัฐบาลประกาศนโยบายเศรษฐกิจใหม่"))